In [ ]:
# -*- coding: utf-8 -*-
# Phylum-ordered BioLog heatmap + fixed-order species dendrogram
# Uses short model prefixes (e.g., "cohm", "mip") on the x-axis.

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage, to_tree, dendrogram
from collections import defaultdict

# --------------------------- CONFIG ---------------------------

file_path = "./SubstarateUtilization&ConfusionMetrics.xlsx"  # adjust as needed

# Model-prefix → phylum mapping (your dict)
phyla_dict = {
    "cohm": "Firmicutes", "cuvk": "Actinobacteria", "cxy":  "Actinobacteria",
    "mip":  "Actinobacteria", "rfa":  "Actinobacteria", "cpi": "Bacteroidetes",
    "flk":  "Bacteroidetes",  "mgk":  "Bacteroidetes",  "bbet": "Alphaproteobacteria",
    "buol": "Betaproteobacteria", "cauj": "Alphaproteobacteria", "dri": "Alphaproteobacteria",
    "lgu":  "Gammaproteobacteria", "lur":  "Gammaproteobacteria", "mlot": "Alphaproteobacteria",
    "pmor": "Gammaproteobacteria", "pscf": "Gammaproteobacteria", "pzu":  "Alphaproteobacteria",
    "racd": "Gammaproteobacteria", "tro":  "Alphaproteobacteria", "vpe":  "Betaproteobacteria",
}

# Desired phylum order (your sequence)
phylum_order = [
    "Bacteroidetes",
    "Firmicutes",
    "Alphaproteobacteria",
    "Betaproteobacteria",
    "Gammaproteobacteria",  # note: double 'm'
    "Actinobacteria",
]
# --------------------------------------------------------------


# ----------------------- LOAD & SHAPE -------------------------

data = pd.read_excel(file_path)

# Match your original indexing
species = data.iloc[1:, 2].astype(str)      # species names
models  = data.iloc[1:, 3].astype(str)      # model names
substrates = data.iloc[0, 4:].astype(str)   # substrate names (header row)
data_values = data.iloc[1:, 4:]             # 0/1 matrix

# Label the matrix with names
data_values.columns = substrates
data_values.index   = species

# Ensure numeric & binary-ish (0/1)
data_values = data_values.apply(pd.to_numeric, errors="coerce").fillna(0).clip(0, 1)

# ------------------- PHYLUM-BASED ORDERING -------------------

phylum_pos = {p: i for i, p in enumerate(phylum_order)}

def _canon_phylum(name: str) -> str:
    if not isinstance(name, str):
        return "Unknown"
    n = name.strip().lower()
    if n in {"alpha", "alphaproteo", "alphaproteobacteria"}:
        return "Alphaproteobacteria"
    if n in {"beta", "betaproteo", "betaproteobacteria"}:
        return "Betaproteobacteria"
    if n in {"gamma", "gammaproteo", "gammaproteobacteria", "gamaproteobacteria"}:
        return "Gammaproteobacteria"
    if n == "bacteroidetes":
        return "Bacteroidetes"
    if n == "firmicutes":
        return "Firmicutes"
    if n == "actinobacteria":
        return "Actinobacteria"
    return name

def model_to_phylum(model_str: str) -> str:
    """Map a model string to a phylum using your phyla_dict (prefix before '_')."""
    if not isinstance(model_str, str):
        return "Unknown"
    tok = model_str.split("_")[0].lower()
    best_key = None
    for k in phyla_dict.keys():
        if tok.startswith(k):
            if best_key is None or len(k) > len(best_key):
                best_key = k
    ph = phyla_dict.get(best_key, "Unknown")
    return _canon_phylum(ph)

# Build an ordering table to avoid issues with duplicates
order_df = pd.DataFrame({
    "Species": species.values,
    "Model":  models.values,
    "orig_pos": range(len(species)),
})
order_df["Phylum"] = order_df["Model"].map(model_to_phylum)
order_df["phylum_rank"] = order_df["Phylum"].map(lambda p: phylum_pos.get(p, len(phylum_order)))

# Warn if some models didn't map
unknown_rows = order_df[order_df["Phylum"] == "Unknown"]
if not unknown_rows.empty:
    print(f"[warn] {len(unknown_rows)} models did not map to a known phylum; placed at the end.")

# Final species order: phylum bucket → original appearance → model
order_df = order_df.sort_values(["phylum_rank", "orig_pos", "Model"], kind="stable")
species_order = order_df["Species"].tolist()

# ----------------------- SHORT DISPLAY NAMES ------------------

# Use short model prefix (before '_'), lowercased; ensure uniqueness (cohm, cohm_2, ...)
species_to_short = dict(zip(order_df["Species"], order_df["Model"].str.split("_").str[0].str.lower()))
short_labels = [species_to_short[s] for s in species_order]
counts = defaultdict(int)
short_labels_unique = []
for lab in short_labels:
    counts[lab] += 1
    if counts[lab] == 1:
        short_labels_unique.append(lab)
    else:
        short_labels_unique.append(f"{lab}_{counts[lab]}")

# ----------------- APPLY ORDER TO THE MATRIX ------------------

data_values_reordered = data_values.loc[species_order]

# --------------------------- HEATMAP --------------------------

plt.figure(figsize=(12, 10))
ax = sns.heatmap(
    data_values_reordered.T,
    cmap="Greys",
    cbar=False,
    linewidths=0.5,
    vmin=0, vmax=1,
)

# use short labels
ax.set_xticklabels(short_labels_unique, rotation=90)
plt.yticks(rotation=0)
plt.title("Heatmap of BioLog Data Ordered by Phylum")

plt.tight_layout()
plt.savefig("biolog_heatmap.pdf", format="pdf", bbox_inches="tight", pad_inches=0.25)
plt.close()
print("Heatmap saved as 'biolog_heatmap.pdf'")

# --------- FIXED-ORDER SPECIES DENDROGRAM (matches heatmap) ---------

# Cluster on species-by-substrate (binary) matrix
X = (data_values.loc[species_order] > 0).astype(int)
D = pdist(X.values, metric="jaccard")
Z = linkage(D, method="average")  # choose any method you like

def plot_dendrogram_fixed_order(Z, labels, target_order, xlabels=None,
                                orientation="top", figsize=(12, 4)):
    """
    Draw a dendrogram using the exact target_order for leaves.
    Z: linkage matrix
    labels: list of labels in the same order as the observations used in pdist
    target_order: desired leaf order (must be a permutation of labels)
    xlabels: labels to display at the leaves (defaults to target_order)
    """
    # checks
    if set(labels) != set(target_order):
        missing = set(labels) - set(target_order)
        extra   = set(target_order) - set(labels)
        raise ValueError(f"target_order must be a permutation of labels. Missing={missing}, Extra={extra}")

    # map label -> fixed x position
    xpos = {lab: i for i, lab in enumerate(target_order)}
    n = len(labels)

    # build tree from linkage
    root = to_tree(Z, rd=False)

    # leaf id (0..n-1) -> label (follows row order of X)
    id2label = {i: labels[i] for i in range(n)}

    fig, ax = plt.subplots(figsize=figsize)

    def walk(node):
        """Return (x,y) of node; draw lines for branches."""
        if node.is_leaf():
            lab = id2label[node.id]
            x = xpos[lab]
            y = 0.0
            return x, y

        xL, yL = walk(node.get_left())
        xR, yR = walk(node.get_right())
        y = node.dist

        # draw lines
        ax.plot([xL, xL], [yL, y], linewidth=1)
        ax.plot([xR, xR], [yR, y], linewidth=1)
        ax.plot([xL, xR], [y,  y], linewidth=1)

        x = (xL + xR) / 2.0
        return x, y

    walk(root)

    # cosmetics
    ax.set_xlim(-0.5, n - 0.5)
    ymax = max(Z[:, 2]) if len(Z) else 1.0
    ax.set_ylim(0, ymax * 1.05)
    if orientation == "top":
        ax.invert_yaxis()
        ax.xaxis.tick_top()
    ax.set_xticks(range(n))
    leaf_labels = xlabels if xlabels is not None else target_order
    ax.set_xticklabels(leaf_labels, rotation=90, ha="center", fontsize=8)
    ax.set_ylabel("Distance")
    ax.set_title("Species clustering dendrogram (fixed to phylum/heatmap order)")
    plt.tight_layout()
    return fig, ax

labels_for_Z = list(species_order)  # same order as X rows
fig, ax = plot_dendrogram_fixed_order(
    Z,
    labels=labels_for_Z,
    target_order=species_order,
    xlabels=short_labels_unique,     # <-- show short names
    orientation="top",
    figsize=(12, 4)
)
fig.savefig("biolog_species_dendrogram_FIXED.pdf", format="pdf", bbox_inches="tight")
plt.close(fig)
print("Species dendrogram saved as 'biolog_species_dendrogram_FIXED.pdf'")

# --------- OPTIONAL: substrate dendrogram (standard order) ---------

Xs = (data_values_reordered.T > 0).astype(int)  # rows = substrates
Ds = pdist(Xs.values, metric="jaccard")
Zs = linkage(Ds, method="average")

plt.figure(figsize=(6, 10))
dendrogram(
    Zs,
    labels=Xs.index.tolist(),
    orientation="right",
    leaf_font_size=8,
)
plt.title("Substrate clustering dendrogram (Jaccard, average linkage)")
plt.tight_layout()
plt.savefig("biolog_substrate_dendrogram.pdf", format="pdf", bbox_inches="tight")
plt.close()
print("Substrate dendrogram saved as 'biolog_substrate_dendrogram.pdf'")
